# Validation set generation

In [13]:
import pandas as pd

train_path = "./dataset_new/train.csv"
test_path = "./dataset_new/TEST_00.csv"
out_path = "./dataset_new/Validation.csv"

train = pd.read_csv(train_path, parse_dates=["date"])
test = pd.read_csv(test_path, parse_dates=["date"])

augmented_list = []

for sm, g_test in test.groupby("store_menu"):
    g_train = train[train["store_menu"] == sm]
    extra = g_train[(g_train["date"] >= "2024-06-09") & (g_train["date"] <= "2024-06-15")]
    g_aug = pd.concat([extra, g_test], ignore_index=True)
    augmented_list.append(g_aug)

test_augmented = pd.concat(augmented_list, ignore_index=True)
test_augmented = test_augmented.sort_values(["store_menu", "date"]).reset_index(drop=True)

test_augmented.to_csv(out_path, index=False)
print(f"Saved: {out_path}")


Saved: ./dataset_new/Validation.csv


# Invalid sales check

In [14]:
import pandas as pd
from pathlib import Path

data_dir = Path("./dataset_new")
csv_files = list(data_dir.glob("*.csv"))

df_list = []
for fp in csv_files:
    tmp = pd.read_csv(fp, parse_dates=["date"])
    tmp["source_file"] = fp.name
    df_list.append(tmp)

df = pd.concat(df_list, ignore_index=True)

invalid = df[(df["sales"] < 0) | (df["sales"] > 1500)]
invalid = invalid[["source_file", "date", "store_menu", "sales"]]

invalid.to_csv("invalid_data.csv", index=False)

print("Total rows:", len(df))
print("Invalid rows:", len(invalid))
print("Saved invalid_data.csv")


Total rows: 163471
Invalid rows: 22
Saved invalid_data.csv


# minus values to zero

In [15]:
import pandas as pd
from pathlib import Path

data_dir = Path("./dataset_new")
csv_files = list(data_dir.glob("*.csv"))

for fp in csv_files:
    df = pd.read_csv(fp, parse_dates=["date"])
    df.loc[df["sales"] < 0, "sales"] = 0
    df.to_csv(fp, index=False)
    print(f"Updated: {fp.name}")


Updated: Validation.csv
Updated: train.csv
Updated: TEST_08.csv
Updated: TEST_03.csv
Updated: TEST_09.csv
Updated: TEST_02.csv
Updated: TEST_00.csv
Updated: TEST_07.csv
Updated: TEST_01.csv
Updated: TEST_05.csv
Updated: TEST_04.csv
Updated: TEST_06.csv
